### Quantum Computing Assignment - BB84 Simulation with Attacker

The aim of the assignment is to simulate the BB84 key distribution protocol.

This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [ ]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math
import numpy as np

In [85]:
simulator = BasicSimulator()
NUM_QUBITS = 38
QBER_SECURITY_THRESHOLD = 15

def get_random_bit(sim):
    """Generates a random bit (0 or 1) by measuring a qubit in superposition."""
    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)
    result = sim.run(qc, shots=1, memory=True).result()
    return int(result.get_memory()[0])

def encode_qubits(bits, bases):
    """Encodes bits onto qubits using the specified bases."""
    encoded_qubits = []
    for i in range(len(bits)):
        qc = QuantumCircuit(1, 1)
        if bits[i] == 1:
            qc.x(0)
        if bases[i] == 1:
            qc.h(0)
        encoded_qubits.append(qc)
    return encoded_qubits

def measure_qubits(qubits, bases, sim):
    """Measures qubits using the specified bases."""
    measured_bits = []
    for i in range(len(qubits)):
        qc = qubits[i].copy()
        if bases[i] == 1:
            qc.h(0)
        qc.measure(0, 0)
        result = sim.run(qc, shots=1, memory=True).result()
        measured_bits.append(int(result.get_memory()[0]))
    return measured_bits

### Alice's Encoding

In [86]:
alice_bits = [get_random_bit(simulator) for _ in range(NUM_QUBITS)]

alice_bases = [get_random_bit(simulator) for _ in range(NUM_QUBITS)]

print(f"Alice's bits:  {np.array(alice_bits)}")
print(f"Alice's bases: {np.array(alice_bases)}")

alice_qubits = encode_qubits(alice_bits, alice_bases)
print("Alice has encoded her qubits and sent them towards Bob.")

Alice's bits:  [1 0 0 1 1 1 0 0 0 1 0 0 1 1 1 1 0 1 0 1 1 0 0 0 0 0 1 0 1 1 0 1 0 1 1 1 1
 0]
Alice's bases: [1 0 0 1 0 1 1 0 1 0 1 1 1 1 0 1 0 0 0 0 1 1 0 0 0 1 0 1 0 1 0 0 0 0 1 1 1
 0]
Alice has encoded her qubits and sent them towards Bob.


### Eve's Interception (Intercept-Resend Attack)

In [87]:
print("--- Eve is Eavesdropping ---")
eve_bases = [get_random_bit(simulator) for _ in range(NUM_QUBITS)]
print(f"Eve's bases: {np.array(eve_bases)}")

eve_bits = measure_qubits(alice_qubits, eve_bases, simulator)
print(f"Eve's measured bits: {np.array(eve_bits)}")

qubits_for_bob = encode_qubits(eve_bits, eve_bases)
print("Eve has sent new qubits to Bob.")

--- Eve is Eavesdropping ---
Eve's bases: [1 0 0 1 1 1 1 0 1 1 0 1 0 0 1 1 1 1 0 0 0 0 0 1 0 0 1 0 1 1 1 0 1 0 0 1 1
 1]
Eve's measured bits: [1 0 0 1 0 1 0 0 0 0 0 0 0 1 1 1 0 0 0 1 0 1 0 1 0 1 0 0 1 1 0 1 0 1 0 1 1
 0]
Eve has sent new qubits to Bob.


### Bob's Measurement

In [88]:
print("\n--- Bob Receives and Measures ---")
bob_bases = [get_random_bit(simulator) for _ in range(NUM_QUBITS)]
print(f"Bob's bases:   {np.array(bob_bases)}")

bob_bits = measure_qubits(qubits_for_bob, bob_bases, simulator)
print(f"Bob's measured bits: {np.array(bob_bits)}")


--- Bob Receives and Measures ---
Bob's bases:   [1 0 1 1 0 0 0 0 1 1 1 1 0 0 0 0 0 0 1 0 0 0 1 0 0 0 1 1 1 0 0 0 1 1 1 0 0
 1]
Bob's measured bits: [1 0 1 1 0 0 1 0 0 0 0 0 0 1 1 0 1 1 1 1 0 1 0 1 0 1 0 0 1 1 0 1 0 0 1 1 1
 0]


### Theoretical Error Rate

A random intercept-resend attack causes ~25% errors in the matching-basis bits (Eve guesses the wrong basis 50% of the time, and when she does, Bob's result is random, causing an error 50% of *that* time). Alice and Bob sacrifice a sample of key bits for checking; if the error rate exceeds a security threshold (e.g., 15%), they conclude the channel is compromised.

### Sifting and Eavesdropper Detection

In [89]:
sifted_key_alice = []
sifted_key_bob = []
for i in range(NUM_QUBITS):
    if alice_bases[i] == bob_bases[i]:
        sifted_key_alice.append(alice_bits[i])
        sifted_key_bob.append(bob_bits[i])

print(f"\n--- Sifting Results (where Alice and Bob's bases matched) ---")
print(f"Alice's sifted key: {np.array(sifted_key_alice)}")
print(f"Bob's sifted key:   {np.array(sifted_key_bob)}")

mismatches = 0
if len(sifted_key_alice) > 0:
    for i in range(len(sifted_key_alice)):
        if sifted_key_alice[i] != sifted_key_bob[i]:
            mismatches += 1
    
    qber = (mismatches / len(sifted_key_alice)) * 100
    print(f"\nNumber of mismatched bits in sifted keys: {mismatches}")
    print(f"Total bits in sifted key: {len(sifted_key_alice)}")
    print(f"Quantum Bit Error Rate (QBER): {qber:.2f}%")
    
    if qber > QBER_SECURITY_THRESHOLD:
        print("\nEavesdropper detected. The QBER is above the security threshold. Key is discarded.")
    else:
        print("\nQBER is below the threshold. This is unusual with an attacker, but possible if Eve got lucky with her basis choices.")
else:
    print("\nNo bits left after sifting. Cannot establish a key.")


--- Sifting Results (where Alice and Bob's bases matched) ---
Alice's sifted key: [1 0 1 1 0 0 0 0 1 0 1 1 0 0 0 0 1 1]
Bob's sifted key:   [1 0 1 0 0 0 0 0 1 1 1 1 1 0 0 0 1 1]

Number of mismatched bits in sifted keys: 3
Total bits in sifted key: 18
Quantum Bit Error Rate (QBER): 16.67%

Eavesdropper detected. The QBER is above the security threshold. Key is discarded.
